In [23]:
import sys,os
sys.path.append(r'C:/Users/andrej/Projects/EnergyTrading/Python/Strategies/LeadLagXGB/')
sys.path.append(r'Z:/EnergyTrading/Python/')
sys.path.append(r'Z:/EnergyTrading/Python/Strategies/LeadLagXGB/')
from support_functions import calculate_MACD, calculate_lead_lag_triggers, calculate_regression_model_price, calc_vol_intensity_index

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass
tol=(1e-1)/2

In [25]:
class EMA:
    def __init__(self, span):
        self.span = span
        self.value = 0
        self.alpha = 2 / (span + 1)

    def push(self, value):
        self.value = self.alpha * value + (1 - self.alpha) * self.value

class TR_class:
    def __init__(self, tau, tau_ema, burn=10):
        self.tau = tau
        self.tau_ema = tau_ema
        self.reset()
        self.__burn = burn

    @property
    def param_keys(self):
        return ['tau', 'tau_ema']

    def update_params(self, params_dict):
        if 'tau' in params_dict.keys():
            self.tau = params_dict['tau']
        if 'tau_ema' in params_dict.keys():
            self.tau_ema = params_dict['tau_ema']

    @property
    def ewma_val(self):
        return self.ewma.value

    @property
    def is_burn(self):
        return self.tot_n < self.burn

    @property
    def min_tau(self):
        return self.tau // 3

    @property
    def old_value(self):
        return self.__old_value

    @property
    def bt(self):
        return self.__bt

    @property
    def burn(self):
        return self.__burn

    @property
    def tot_n(self):
        return self.__tot_n

    def reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index = 0
        self.__bt = 1
        self.__old_value = np.nan
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def soft_reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index += 1
        self.__bt = 1
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def initialize(self):
        self.ewma.value = .5
        self.ewma_T.value = self.tau
        self.init = False

    def push(self, value, volume=None):
        if self.tot_n < 1:
            self.init = True
        else:
            diff_value = value - self.old_value
            self.__bt = self.signed_tick_vals(diff_value)
            bt = max(self.__bt, 0)
            if self.init:
                self.initialize()
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
            if self.is_burn:
                self.phiT += bt
                self.__n += 1
            elif max(self.phiT, self.__n - self.phiT) < self.thres:
                self.phiT += bt
                self.__n += 1
            else:
                self.ewma.push(self.phiT / self.__n)
                self.ewma_T.push(self.__n)
                self.phiT = 0
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
                self.index += 1
                self.__n = 0
        self.__tot_n += 1
        self.__old_value = value
        return self.index

    def signed_tick_vals(self, diff_value):
        if diff_value > 0:
            return 1
        elif diff_value < 0:
            return -1
        else:
            return 0

    def tick_imbalance_single(self, trades):
        self.soft_reset()
        index_series = []
        for trade in trades:
            value = trade[0]
            if not value or np.isnan(value):
                index_series.append((trade[2], self.index))
            else:
                index_series.append((trade[2], self.push(value)))

        return index_series

    def tick_imbalance_indices(self, trades):
        self.reset()
        index_series = []
        current_date = None
        daily_trades = []

        for trade in trades:
            trade_date = trade[2].date()
            if current_date is None:
                current_date = trade_date

            if trade_date != current_date:
                # Process the previous day's trades
                index_series.extend(self.tick_imbalance_single(daily_trades))
                daily_trades = []
                current_date = trade_date

            daily_trades.append(trade)

        # Process the last day's trades
        if daily_trades:
            index_series.extend(self.tick_imbalance_single(daily_trades))

        return index_series

def get_nine_am_unix_today_cet():
    # Define the CET timezone
    cet = pytz.timezone('CET')

    # Get today's date in the CET timezone
    today = datetime.now(cet).date()

    # Combine today's date with the time 09:00 AM in CET
    nine_am_today = cet.localize(datetime.combine(today, time(9, 0)))

    # Convert to Unix timestamp (seconds since epoch)
    unix_timestamp = int(nine_am_today.timestamp())

    return unix_timestamp

def calculate_ema(current_price, previous_ema, span):
    alpha = 2 / (span + 1)
    return alpha * current_price + (1 - alpha) * previous_ema

def get_trades_for_contract(contract, start_date, end_date):
    params_dict={}
    params_dict['tenor_list'] = ['dec'] if contract=='euadec1' else [contract[-2]]
    params_dict['tn1_list'] = [int(contract[-1])]
    params_dict['mkt_list'] = ['eua'] * len(params_dict['tenor_list']) if contract=='euadec1' else [contract[0:-2]] * len(params_dict['tenor_list'])
    params_dict['tn2_list'] = []
    params_dict['prod'] = 'base'
    params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
    params_dict['start_date'] = start_date
    params_dict['end_date'] = end_date
    params_dict['ns'] = 2

    # Fetch trades and best orders for the curve
    assembler = TPDataAssembly(source='trayport', user='matej')
    # assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
    trades_dict = assembler.get_data(params_dict, target_data='trades')
    #assembler.set_data_source('database')
    #ba_dict = assembler.get_data(params_dict, target_data='best_orders')


    trades = pd.DataFrame()
    products = []
    for key in trades_dict.keys():
        trade_aux = trades_dict[key].copy()
        trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
        if trades.empty:
            trades = trade_aux.copy()
        else:
            trades = pd.concat([trades, trade_aux])
        products.append(key)
    trades.sort_index(inplace=True)




    data_raw = trades
    print(data_raw.columns)
    data_raw['tradeid_'+contract]=data_raw['tradeid_'+contract].apply(lambda x: str(x)[:-7] if str(x)[-7:]==' Public' else str(x))
    df_lead = data_raw[data_raw['broker_id_'+contract]==1441][['tradeid_'+contract,'price_'+contract, 'volume_'+contract]].copy()
    
    df_lead['contract']=contract

    # data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
    #                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

    df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
    print(df_lead.columns)
    df_lead.columns = ['tradeid', 'trd_price', 'volume', 'contract']
    
    return df_lead

def fit_model(lead_contract, lag_contract):
    data_lead_trds = df[df['contract']==lead_contract]
    data_lag_trds = df[df['contract']==lag_contract]

    data_lead_trds['tag'] = 'lead'
    data_lag_trds['tag'] = 'lag'

    df_trds = pd.concat([data_lead_trds, data_lag_trds]).sort_index()

    df_trds['lead_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lead' else None, axis=1)
    df_trds['lead_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lead' else None, axis=1)
    df_trds['lead_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lead' else None, axis=1)

    df_trds['lag_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lag' else None, axis=1)
    df_trds['lag_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lag' else None, axis=1)
    df_trds['lag_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lag' else None, axis=1)



    agg_dict = {'index': 'first', 'datetime': 'first'}
    agg_dict.update({k: 'sum' for k in ['lead_volume', 'lead_pv', 'lag_volume', 'lag_pv']})

    df_trds['date'] = df_trds.index.date
    dates_list = sorted(list(set(df_trds['date'])))

    df_list = []
    df_list2 = []

    sort_order=[True, False, True]


    for current_day in dates_list:
        df_trds_1day = df_trds[df_trds['date'] == current_day].reset_index()
        df_trds_1day['execution_time'] = df_trds_1day['datetime'].astype('int64')  # Already in nanoseconds

        # Preparing tick data
        ti_cls = TR_class(tau=10, tau_ema=10)

        df_trds_1day = df_trds_1day.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index()


        # Create the list of tuples
        lag_trades = [(row['lag_price'], row['volume'], row['execution_time']) for index, row in
                      df_trds_1day.iterrows()]


        idx_series = ti_cls.tick_imbalance_single(lag_trades)
        idx_series = pd.DataFrame(idx_series, columns=['index', 0])


        if 'level_0' in df_trds_1day.columns:
            del df_trds_1day['level_0']

        # Create returns
        df_trds_indexed_1day = pd.concat([df_trds_1day, idx_series[0]], axis=1).reset_index()
        lag_index = df_trds_indexed_1day[df_trds_indexed_1day['tag'] == 'lag'].index.min()
        lead_before_lag = df_trds_indexed_1day[(df_trds_indexed_1day['tag'] == 'lead') & (df_trds_indexed_1day.index < lag_index)]
        df_trds_indexed_1day.loc[lead_before_lag.index, 0] = 0

        df_trds_indexed_1day['tick_id'] = str(current_day) + '_' + df_trds_indexed_1day[0].fillna(0).apply(str)
        #df_trds_indexed_1day['datetime'] = df_trds_indexed_1day['timestamp']
        #df_trds_indexed_1day = df_trds_indexed_1day.set_index('datetime')
        df_trds_indexed_1day_grouped = df_trds_indexed_1day.groupby('tick_id').agg(
            agg_dict).reset_index().set_index('datetime')

        df_trds_indexed_1day_grouped['lead_price'] = df_trds_indexed_1day_grouped['lead_pv'] / \
                                                     df_trds_indexed_1day_grouped['lead_volume']
        df_trds_indexed_1day_grouped['lag_price'] = df_trds_indexed_1day_grouped['lag_pv'] / \
                                                    df_trds_indexed_1day_grouped['lag_volume']

        df_trds_indexed_1day_grouped['lead_log_ret'] = np.log(df_trds_indexed_1day_grouped['lead_price'].ffill() ).diff()
        df_trds_indexed_1day_grouped['lag_log_ret'] = np.log(df_trds_indexed_1day_grouped['lag_price'].ffill() ).diff()
        df_list.append(df_trds_indexed_1day_grouped)
        df_list2.append(df_trds_indexed_1day)

    df_trds_indexed = pd.concat(df_list).sort_index()
    df_trds = pd.concat([df_trds.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True), pd.concat(df_list2).sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True)['tick_id']], axis=1)

    #Model fitting and scoring
    # Prepare to store predictions
    df_trds_indexed['lag_log_ret_pred'] = np.nan
    df_trds_indexed['coef1'] = np.nan
    df_trds_indexed['coef2'] = np.nan
    df_trds_indexed['date'] = df_trds_indexed.index.date
    dates_list = sorted(list(set(df_trds_indexed['date'])))    
    
    training_data = df_trds_indexed
    # Skip if not enough data


    # Independent variable (lead_log_ret) and dependent variable (lag_log_ret)
    X_train = training_data['lead_log_ret'].fillna(0)
    y_train = training_data['lag_log_ret'].fillna(0)

    # Add a constant to the independent variable
    X_train = sm.add_constant(X_train)

    # Fit the model using statsmodels
    model = sm.OLS(y_train, X_train).fit()
    
    return model.params[0], model.params[1], model.pvalues[0], model.pvalues[1], model.rsquared

In [26]:
contract_pairs=[
                #('deq1','dem2'),
                #('ttfm1', 'dem2'),
                ('dem1','dem2'),
                ('dem1','deq1'),
                #('dem2','deq1'),
                #('ttfm1','deq1'),
                                
                #('ttfm1', 'dey1'),
                #('dem1','dey1'),
                #('dem2','dey1'), 
                #('deq1','dey1'),
    
                #('dem1','ttfm1'),
                #('dem2','ttfm1'), 
                #('deq1','ttfm1')
]

pred_id_mapping={
    f'fair_price_{lead}_{lag}': f'll_{lead}_{lag}_fair_price'
    for lead, lag in contract_pairs
}

start_date='2024-12-01'
end_date='2025-05-01'

# ALTERNATIVE PRICES - MAREK WITH AUXILLIARY MARKETS

## Selecting all trades since 2024

In [27]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='2025-01-01' and datetime<='2025-05-12' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 1829270 rows from source database.


In [28]:
# -*- coding: utf-8 -*-
"""
Created on Tue May 13 09:49:17 2025

@author: Marek
"""

from datetime import datetime, time
import pandas as pd
import numpy as np
from Math.lm_class import OnlineScoring
from Database.TPData import TPData, TPDataDa
from Math.ti_class import TR_class
from Strategies.LeadLagEns.lead_lag_ensamble import LMOnline, LMOffline, DataClass
from Utilities.excel_loaders import conn_out_xload


def data_process(df_data_p, tau, tau_ema, df_data_v=None, log_bool=True):
    agg_dict = {'index': 'first'}
    agg_dict.update({k: 'mean' for k in df_data_p.columns})
    ti_cls = TR_class(tau, tau_ema)
    # Indices based on ticks
    idx_series = ti_cls.tick_imbalance_indices(df_data_p.iloc[:, 0])
    # Create returns
    data_aux = pd.concat([df_data_p.reindex(idx_series.index), idx_series], axis=1).reset_index()
    data_aux = data_aux.groupby(0).agg(agg_dict).set_index('index')
    grouped = data_aux.groupby(data_aux.index.date)
    data_dict = {date: group for date, group in grouped}
    df_ret = pd.DataFrame([])
    for data in data_dict.values():
        y_series = data.shift(periods=0).dropna()
        data = data.iloc[:, 0].reindex(y_series.index)
        if log_bool:
            ret_aux = np.log(data.ffill()).diff()
        else:
            ret_aux = data.ffill().diff()
        if df_ret.empty:
            df_ret = ret_aux.dropna()
        else:
            df_ret = pd.concat([df_ret, ret_aux.dropna()])    
    return df_ret

def calc_vol(df_ret):
    grouped = df_ret.groupby(df_ret.index.date)
    data_dict = {date: group for date, group in grouped}
    vol_dict = {date: np.std(vals) for date, vals in data_dict.items()}
    return pd.Series(vol_dict)

def hilovol(data_p_, data_v_, mkt, quant=.6):
    log_bool = True
    # Expected size of candle
    tau = 25
    # EMA
    tau_ema = 25
    df_ret = data_process(data_p_.loc[:, [mkt]], tau, tau_ema, data_v_, log_bool)
    df_vol = calc_vol(df_ret)

    period_list = [3, 10, 21]
    name_list = ['s', 'm', 'h']
    ma_dict = {}
    ma_dict_ = {}
    for n, w in zip(name_list, period_list):
        ma_dict[n] = df_vol.shift(periods=1).rolling(window=w, min_periods=w).mean()
        ma_dict_[n] = df_vol.shift(periods=0).rolling(window=w, min_periods=w).mean()

    df_ma = pd.DataFrame(ma_dict).dropna()
    ma_dict = {m: v.reindex(df_ma.index) for m, v in ma_dict.items()}
    hilo_new = [False for x in df_vol.values]
    for n in name_list:
        diff_quant = (df_vol - df_ma[n]).quantile(quant)
        hilo_new = [True if (x > diff_quant) or b else False for x, b in zip((df_vol - df_ma[n]).values, hilo_new)]
    hilo_new_ser = pd.Series(hilo_new, index=df_vol.index)
    # Filter data
    true_dates = pd.to_datetime(hilo_new_ser[hilo_new_ser].index)
    idx = data_p_.index.normalize().isin(true_dates)
    # Filtered dates
    dates_fltr = pd.DatetimeIndex(hilo_new_ser[hilo_new_ser].index)
    return data_p_.loc[idx, :], data_v_.loc[idx, :], dates_fltr

In [29]:
data_class = TPData()
data_class.create_connection('OracleSQL')
data_class_pg = TPData()
data_class_pg.create_connection('PostgreSQL')
data_class_tp = TPDataDa()

mkt_list = ['de', 'de', 'de', 'de']
tenor_list = ['m', 'q', 'm', 'y']
tn_list = [1, 1, 2, 1]
# mkt_list = ['ttf', 'eua']
# tenor_list = ['q', 'dec']
# tn_list = [2, 1]
prod = 'base'
venue_list = ['eex']
# start_date = datetime(2024, 6, 3)
start_date = datetime(2025, 1, 1)
end_date = datetime(2025, 5, 12)
dates_out = [x.date() for x in conn_out_xload()]

allwd_broker_ids = [1441]

sample_dates = pd.date_range(start_date, end_date, freq='B')

n = 11
n_t = 10
d_t = 1
date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B')
                   for k in sample_dates[:-n+1]}

n_s = 2

dates = pd.date_range(start_date, end_date, freq='B')
product_date = [dates.shift(1, freq='B') if t == 'da' else
                dates.shift(1, freq='D') if t == 'd' else
                dates.shift(tn, freq='W-MON') if t == 'w' else
                (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                for t, tn in zip(tenor_list, tn_list)]

start_time = time(9, 0, 0)
end_time = time(18, 0, 0)

tr_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_list, tenor_list, tn_list)}
agg_dict = {'price': 'sum', 'volume': 'sum', 'action': 'median',
            'broker_id': 'median', 'count': 'sum', 'tradeid': 'first'}

for m, t, n, p_dates in zip(mkt_list, tenor_list, tn_list, product_date):
    df_tr = pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # Trades
        df_tr_aux = data_class.get_trades(m, t, venue_list, p_d, bT, eT, prod)
        # Filter by broker
        if not allwd_broker_ids or t == 'da':
            pass
        else:
            df_tr_aux = df_tr_aux[df_tr_aux['broker_id'].isin(allwd_broker_ids)]
        try:
            df_tr_aux = df_tr_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Group trades
        df_tr_aux['count'] = 1
        df_tr_aux['price'] *= df_tr_aux['volume']
        df_tr_aux = df_tr_aux.groupby(df_tr_aux.index).agg(agg_dict)
        df_tr_aux['price'] /= df_tr_aux['volume']
        df_tr = pd.concat([df_tr, df_tr_aux])
        del df_tr_aux
    tr_data_dict[m + t + str(n)] = df_tr

data_p_ = pd.concat({k: v['price'] for k, v in tr_data_dict.items()}, axis=1)
data_v_ = pd.concat({k: v['volume'] for k, v in tr_data_dict.items()}, axis=1)


#### Auxiliary ###
mkt_aux_list = ['ttf', 'ttf', 'eua']
tenor_aux_list = ['m', 'q', 'dec']
tn_aux_list = [1, 1, 1]

mid_data_dict = {m + t + str(n): [] for m, t, n in zip(mkt_aux_list, tenor_aux_list, tn_aux_list)}
# Auxiliary regressors
product_aux_date = [dates.shift(1, freq='B') if t == 'da' else
                    dates.shift(1, freq='D') if t == 'd' else
                    dates.shift(tn, freq='W-MON') if t == 'w' else
                    (dates + n_s * dates.freq).shift(tn, freq='2QS-Apr') if t in ['sum', 'win'] else
                    (dates + n_s * dates.freq).shift(tn, freq='YS') if t in ['dec'] else
                    (dates + n_s * dates.freq).shift(tn, freq=t.upper() + 'S')
                    for t, tn in zip(tenor_aux_list, tn_aux_list)]

for m, t, n, p_dates in zip(mkt_aux_list, tenor_aux_list, tn_aux_list, product_aux_date):
    df_ba = pd.DataFrame([])
    series = pd.Series(p_dates, index=dates)
    for p_d, ds in series.groupby(series).groups.items():
        bT = datetime.combine(ds[0], start_time)
        eT = datetime.combine(ds[-1], end_time)
        # OB data
        data_class.create_connection('PostgreSQL')
        df_ba_aux = data_class.get_best_ob_data(m, t, venue_list, p_d, bT, eT, prod, None, False)
        df_ba_aux = df_ba_aux.rename(columns={'bidbestprice': 'b_price', 'askbestprice': 'a_price'})
        try:
            df_ba_aux = df_ba_aux.between_time(start_time, end_time)
        except(TypeError):
            pass
        # Omit data from drop dates
        df_ba_aux['date'] = df_ba_aux.index.date
        df_ba_aux = df_ba_aux[~df_ba_aux['date'].isin(dates_out)].drop(columns='date')
        # Filter timestamps
        df_tr_aux = data_p_.loc[bT:eT, :]
        ts = df_tr_aux.index.union(df_ba_aux.index)
        df_ba_aux = df_ba_aux.reindex(ts).ffill().loc[df_tr_aux.index, :]
        df_ba = pd.concat([df_ba, df_ba_aux])
        del df_ba_aux, df_tr_aux
    mid_data_dict[m + t + str(n)] = .5 * (df_ba['b_price'] + df_ba['a_price'])

if mkt_aux_list:
    data_aux = pd.concat({k + '_aux': v for k, v in mid_data_dict.items()}, axis=1)
    # Merge
    data_p_ = pd.concat([data_p_, data_aux], axis=1)
#######

Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database oracle
Disconnected from the database oracle
Connected to the database postgre
Disconnected from the database postgre
Conn

In [30]:
# Data properties
mkt = 'dem1'
mkt_l = ['deq1']#['dem1', 'dem2', 'dey1']
mkt_aux = [x + '_aux' for x in ['ttfm1', 'ttfq1', 'euadec1']]
# mkt = 'ttfq2'
# mkt_l = ['euadec1']
tau = 10
tau_ema = 10
n_ema = 16

diff_bool = True
vol_bool = False
scale_bool = True
isEma = False

tick_val = 200

km_bool = False

# Score
score_class = OnlineScoring('r2', burn=0, tau=200)

# Model params & class
intercept = False
lambda1=.2
lambda2=.0
alpha=17.8
beta=2.9

adaptive_l1 = False
norm_g = False

model_single = LMOnline(1, intercept, lambda1, lambda2, alpha, beta,
                        adaptive_l1, norm_g)

model_type = 'ridge'
alpha = np.logspace(-3, 1, 50)
cv_bool = True
cv = 5
model_single = LMOffline(intercept, model_type=model_type, alpha=alpha,
                         cv_bool=cv_bool, cv=cv)
# Data preparation
model_data = DataClass(mkt, tau, tau_ema, diff_bool, vol_bool, scale_bool,
                       isEma, n_ema)

# Hilo VOL
vol_filter = False
if vol_filter:
    n = 16
    quant = .6
    data_p, data_v, dates_fltr = hilovol(data_p_, data_v_, 'dem1', quant)
    date_range_dict = {k.date(): dates_fltr[i:i+n].difference(dates_out)
                       for i, k in enumerate(dates_fltr[:-n+1]) if k.date() not in dates_out}
else:
    n = 16
    data_p, data_v = data_p_, data_v_
    date_range_dict = {k.date(): pd.date_range(k, periods=n, freq='B').difference(dates_out)
                       for k in sample_dates[:-n+1] if k.date() not in dates_out}
    

data_dict = model_data.data_process_3(data_p, None, km_bool=km_bool, reg_aux=mkt_aux)
split_dict = model_data.split_data(data_dict, date_range_dict, d_t)
# Loop for models
pred_dict = {m: [] for m in mkt_l}
for m in mkt_l:
    # Scale
    reg_data = model_data.scale_data_dict(split_dict[m])
    pred_dict[m] = model_data.prepare_model_trds(data_p, mkt, model_single, reg_data)


db_data = model_data.process_model_db_n(pred_dict, tr_data_dict)

In [31]:
def reorder_fair_ret_tag(s):
    parts = s.split('_')
    
    # Basic safety check
    if len(parts) < 6 or parts[0:2] != ['ll', 'aux']:
        raise ValueError("Unexpected format. Expected: ll_aux_<tag1>_<tag2>_<dem>_<deq>")
    
    prefix = parts[:2]             # ['ll', 'aux']
    fair_ret = parts[2:4]          # ['fair', 'ret']
    dem_deq = parts[4:]            # ['dem1', 'deq1']
    
    reordered = prefix + dem_deq + fair_ret
    return '_'.join(reordered)

db_data.columns=['tradeid']+[reorder_fair_ret_tag(col.replace('price_hat', 'll_aux_fair_price').replace('ret', 'll_aux_fair_ret')+'_'+mkt) for col in db_data.columns[1:]]

In [32]:
db_data.head()

,tradeid,ll_aux_deq1_dem1_fair_price,ll_aux_deq1_dem1_fair_ret
index,,,
2025-01-22 09:00:51.337322734,Eurex T7/DEBQ042025-20250122/778/1,111.076325,-0.213665
2025-01-22 09:01:45.112508804,Eurex T7/DEBQ042025-20250122/799/1,111.040029,-0.517283
2025-01-22 09:02:36.510815582,Eurex T7/DEBQ042025-20250122/811/1,111.022859,-0.660948
2025-01-22 09:02:42.051610668,Eurex T7/DEBQ042025-20250122/816/2,111.027874,-0.057865
2025-01-22 09:03:29.715809681,Eurex T7/DEBQ042025-20250122/827/1,111.056320,0.180119


## Leadlag_fair_price and Saving into TimescaleDB


In [33]:
from itertools import product


for lead_mkt in mkt_l:
    df_merged = df.copy()


    df_merged = df_merged.merge(
    db_data[['tradeid']+[col for col in db_data.columns if lead_mkt in col]].dropna(),
    on='tradeid', how='left'
    )

    df_long = df_merged.melt(
        id_vars=['datetime', 'nanotime', 'tradeid'],  # Keep these columns fixed
        value_vars=[col for col in db_data.columns if lead_mkt in col],
        var_name='pred_name',
        value_name='pred_value'
    )    
    # Step 2: Add pred_name and additional columns if required
    df_long['pred_id'] = df_long['pred_name']# or use a mapping dict if needed
    df_long['additional'] = None  # set accordingly if you have additional data

    # Step 3: Sort by datetime for proper forward filling
    df_long = df_long.sort_values(by=['datetime', 'nanotime','pred_id'], ascending=[True, True, True])

    # Step 4: Forward fill within each day separately
    df_long['date_only'] = df_long['datetime'].dt.date
    df_long['pred_value'] = df_long.groupby(['pred_id', 'date_only'])['pred_value'].ffill()

    # Optionally drop rows where pred_value is still NaN after forward fill
    #df_long = df_long.dropna(subset=['pred_value'])

    # Step 5: Drop helper columns
    df_long = df_long.drop(columns=['date_only'])

    # Locate rows where pred_value is missing and set additional message
    mask_missing = df_long['pred_value'].isna()
    df_long.loc[mask_missing, 'additional'] = 'Not enough observations to calculate the pred_value!'


    # Optional step: enforce data types explicitly
    df_long['pred_value'] = df_long['pred_value'].astype(float)

    df_long=df_long[['datetime', 'nanotime', 'tradeid', 'pred_name', 'pred_id','pred_value', 
           'additional']].reset_index(drop=True)
    
    # 2. Connect to TimescaleDB
    batch_size=100_000
    conn = Database('timescaledb')
    conn._connect()

    # 3. Insert in batches
    total_rows = len(df_long)
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch = df_long.iloc[start:end]

        batch.to_sql('experimental_dataset_entries', conn.engine,schema='public', index=False, if_exists='append',method='multi')
        print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

print("🎉 All batches inserted successfully.")

Connected to the database timescaledb
✅ Inserted rows 0 to 100000 into TimescaleDB.
✅ Inserted rows 100000 to 200000 into TimescaleDB.
✅ Inserted rows 200000 to 300000 into TimescaleDB.
✅ Inserted rows 300000 to 400000 into TimescaleDB.
✅ Inserted rows 400000 to 500000 into TimescaleDB.
✅ Inserted rows 500000 to 600000 into TimescaleDB.
✅ Inserted rows 600000 to 700000 into TimescaleDB.
✅ Inserted rows 700000 to 800000 into TimescaleDB.
✅ Inserted rows 800000 to 900000 into TimescaleDB.
✅ Inserted rows 900000 to 1000000 into TimescaleDB.
✅ Inserted rows 1000000 to 1100000 into TimescaleDB.
✅ Inserted rows 1100000 to 1200000 into TimescaleDB.
✅ Inserted rows 1200000 to 1300000 into TimescaleDB.
✅ Inserted rows 1300000 to 1400000 into TimescaleDB.
✅ Inserted rows 1400000 to 1500000 into TimescaleDB.
✅ Inserted rows 1500000 to 1600000 into TimescaleDB.
✅ Inserted rows 1600000 to 1700000 into TimescaleDB.
✅ Inserted rows 1700000 to 1800000 into TimescaleDB.
✅ Inserted rows 1800000 to 19000

# ALTERNATIVE PRICES BASED ON LL PRODUCTION MODEL

## Selecting all trades since 2024

In [18]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='2025-05-01' and datetime<='2025-05-12' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 103108 rows from source database.


## Selecting all lead and lag EEX trades since 2024 -> this is the base for predictor calculations

In [19]:
contracts=list(set([x[0] for x in contract_pairs]).union(set([x[1] for x in contract_pairs])))
contracts

['deq1', 'dem2', 'dem1']

In [7]:
df_all=pd.concat([get_trades_for_contract(contract, start_date, end_date) for contract in contracts])

https://referencedata.trayport.com/instruments
Duration: 0.750s
https://analytics.trayport.com/api/trades?from=2024-12-02T07%3A00%3A00Z&until=2024-12-27T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=85&ContractType=SinglePeriod
Duration: 0.741s
https://referencedata.trayport.com/instruments
Duration: 0.753s
https://analytics.trayport.com/api/trades?from=2024-12-30T07%3A00%3A00Z&until=2025-01-31T07%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=86&ContractType=SinglePeriod
Duration: 0.846s
https://referencedata.trayport.com/instruments
Duration: 0.753s
https://analytics.trayport.com/api/trades?from=2025-01-31T07%3A00%3A00Z&until=2025-03-04T07%3A00%3A00Z&instrumentId=10641710&sequenceId=10000105&sequenceItemId=86&ContractType=SinglePeriod
Duration: 1.232s
https://referencedata.trayport.com/instruments
Duration: 0.595s
https://analytics.trayport.com/api/trades?from=2025-03-04T07%3A00%3A00Z&until=2025-03-27T19%3A00%3A00Z&instrumentId=10641710&s

In [8]:
df_all['contract'].value_counts()

contract
dem1    303858
dem2    115051
deq1     85675
Name: count, dtype: int64

In [9]:
df_all=df_all.reset_index()

In [10]:
df_all['datetime'] = pd.to_datetime(df_all['datetime'], errors='coerce')
df_all['timestamp'] = df_all['datetime']

In [11]:
df_all=df_all[df_all['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]

In [12]:
df_all.head()

,datetime,tradeid,trd_price,volume,contract,timestamp
77,2024-12-02 09:01:23.495017767,Eurex T7/DEBQ012025-20241202/564/1,111.37,1,deq1,2024-12-02 09:01:23.495017767
78,2024-12-02 09:01:23.503465414,Eurex T7/DEBQ012025-20241202/565/1,111.35,1,deq1,2024-12-02 09:01:23.503465414
79,2024-12-02 09:01:35.834584713,Eurex T7/DEBQ012025-20241202/572/1,111.35,1,deq1,2024-12-02 09:01:35.834584713
80,2024-12-02 09:02:15.867837667,Eurex T7/DEBQ012025-20241202/574/1,111.35,1,deq1,2024-12-02 09:02:15.867837667
81,2024-12-02 09:02:16.189331293,Eurex T7/DEBQ012025-20241202/576/1,111.35,1,deq1,2024-12-02 09:02:16.189331293


## Leadlag_fair_price and Saving into TimescaleDB


In [13]:
from itertools import product

df_merged = df.copy()


df_merged = df_merged.merge(
db_data,
on='tradeid', how='left'
)

# Step 1: Convert wide df to long format using melt
df_merged['date_only'] = df_merged['datetime'].dt.date
df_merged[col_name] = df_merged.groupby(['date_only'])[col_name].ffill()

del df_merged['date_only']
    
# 2. Connect to TimescaleDB
batch_size=100_000
conn = Database('timescaledb')
conn._connect()

# 3. Insert in batches
total_rows = len(df_merged)
for start in range(0, total_rows, batch_size):
    end = min(start + batch_size, total_rows)
    batch = df_merged.iloc[start:end]

    batch.to_sql('alternative_fair_prices', conn.engine,schema='user_andrej_zubal', index=False, if_exists='append',method='multi')
    print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

print("🎉 All batches inserted successfully.")

DEM1 DEM2 2 10 3
DEM1 DEM2 4 10 3
DEM1 DEM2 6 10 3
DEM1 DEM2 8 10 3
DEM1 DEM2 10 10 3
DEM1 DEQ1 2 10 3
DEM1 DEQ1 4 10 3
DEM1 DEQ1 6 10 3
DEM1 DEQ1 8 10 3
DEM1 DEQ1 10 10 3
Connected to the database timescaledb
✅ Inserted rows 0 to 100000 into TimescaleDB.
✅ Inserted rows 100000 to 200000 into TimescaleDB.
✅ Inserted rows 200000 to 300000 into TimescaleDB.
✅ Inserted rows 300000 to 400000 into TimescaleDB.
✅ Inserted rows 400000 to 500000 into TimescaleDB.
✅ Inserted rows 500000 to 600000 into TimescaleDB.
✅ Inserted rows 600000 to 700000 into TimescaleDB.
✅ Inserted rows 700000 to 800000 into TimescaleDB.
✅ Inserted rows 800000 to 900000 into TimescaleDB.
✅ Inserted rows 900000 to 1000000 into TimescaleDB.
✅ Inserted rows 1000000 to 1100000 into TimescaleDB.
✅ Inserted rows 1100000 to 1200000 into TimescaleDB.
✅ Inserted rows 1200000 to 1300000 into TimescaleDB.
✅ Inserted rows 1300000 to 1400000 into TimescaleDB.
✅ Inserted rows 1400000 to 1500000 into TimescaleDB.
✅ Inserted rows 150